# 04 · Análisis exploratorio con SQL

**Objetivo:** responder las 10 preguntas del laboratorio con SQL sobre la tabla `SPACEXTABLE`.

Se usa **SQLite** (incluido en Python) para que el notebook funcione sin servidor. Las consultas son SQL
estándar y funcionan igual en IBM Db2 con cambios mínimos.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [ ]:
import json
import re
import sqlite3

spacex = load_csv("Spacex.csv", f"{IBM}/labs/module_2/data/Spacex.csv")

# Normaliza la fecha a ISO (AAAA-MM-DD) para poder filtrar por rangos y extraer año/mes con substr().
def to_iso(value: str) -> str:
    value = str(value).strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return value
    return pd.to_datetime(value, dayfirst=True).strftime("%Y-%m-%d")

spacex["Date"] = spacex["Date"].map(to_iso)
spacex.columns = [c.strip() for c in spacex.columns]

con = sqlite3.connect(":memory:")
spacex.to_sql("SPACEXTBL", con, if_exists="replace", index=False)
con.execute("DROP TABLE IF EXISTS SPACEXTABLE")
con.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")
print(pd.read_sql("SELECT COUNT(*) AS filas FROM SPACEXTABLE", con))
pd.read_sql("SELECT * FROM SPACEXTABLE LIMIT 3", con)

In [ ]:
sql_results = []


def run(task: str, query: str) -> pd.DataFrame:
    """Ejecuta la consulta, la muestra y la guarda en results/sql_results.json."""
    result = pd.read_sql(query, con)
    sql_results.append({
        "task": task,
        "query": " ".join(query.split()),
        "columns": list(result.columns),
        "rows": result.astype(str).values.tolist(),
    })
    return result

### Tarea 1 · Nombres únicos de los sitios de lanzamiento

In [ ]:
run("Sitios de lanzamiento únicos", """
SELECT DISTINCT Launch_Site
FROM SPACEXTABLE
""")

### Tarea 2 · 5 registros cuyo sitio empieza por 'CCA'

In [ ]:
run("Sitios que empiezan por 'CCA'", """
SELECT Date, Booster_Version, Launch_Site, Payload, PAYLOAD_MASS__KG_
FROM SPACEXTABLE
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5
""")

### Tarea 3 · Masa total de carga transportada para la NASA (CRS)

In [ ]:
run("Masa total de carga NASA (CRS)", """
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_kg
FROM SPACEXTABLE
WHERE Customer = 'NASA (CRS)'
""")

### Tarea 4 · Masa media de carga del booster F9 v1.1

In [ ]:
run("Masa media de carga F9 v1.1", """
SELECT ROUND(AVG(PAYLOAD_MASS__KG_), 2) AS Avg_Payload_Mass_kg
FROM SPACEXTABLE
WHERE Booster_Version = 'F9 v1.1'
""")

### Tarea 5 · Fecha del primer aterrizaje exitoso en plataforma terrestre

In [ ]:
run("Primer aterrizaje exitoso en tierra", """
SELECT MIN(Date) AS First_Successful_Ground_Landing
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (ground pad)'
""")

### Tarea 6 · Boosters con aterrizaje exitoso en barcaza y carga entre 4.000 y 6.000 kg

In [ ]:
run("Éxito en barcaza con carga 4.000–6.000 kg", """
SELECT Booster_Version
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000
  AND PAYLOAD_MASS__KG_ < 6000
""")

### Tarea 7 · Número total de misiones exitosas y fallidas

In [ ]:
run("Resultados de misión", """
SELECT TRIM(Mission_Outcome) AS Mission_Outcome, COUNT(*) AS Total
FROM SPACEXTABLE
GROUP BY TRIM(Mission_Outcome)
""")

### Tarea 8 · Boosters que transportaron la carga máxima (subconsulta)

In [ ]:
run("Boosters con la carga máxima", """
SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)
""")

### Tarea 9 · Fallos de aterrizaje en barcaza durante 2015 (mes, booster y sitio)

In [ ]:
run("Fallos en barcaza en 2015", """
SELECT substr(Date, 6, 2) AS Month, Date, Booster_Version, Launch_Site, Landing_Outcome
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Failure (drone ship)'
  AND substr(Date, 1, 4) = '2015'
""")

### Tarea 10 · Ranking de resultados de aterrizaje entre 2010-06-04 y 2017-03-20

In [ ]:
run("Ranking de aterrizajes 2010-06-04 a 2017-03-20", """
SELECT Landing_Outcome, COUNT(*) AS Total
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Total DESC
""")

In [ ]:
(RESULTS_DIR / "sql_results.json").write_text(json.dumps(sql_results, indent=2, ensure_ascii=False))
con.close()
print(f"{len(sql_results)} consultas guardadas en results/sql_results.json")

## Conclusiones del análisis SQL

* SpaceX opera desde 4 plataformas (dos en Cabo Cañaveral, una en Kennedy y una en Vandenberg).
* El primer aterrizaje exitoso en tierra llega años después del primer lanzamiento: la recuperación fue
  un proceso de aprendizaje.
* Casi todas las **misiones** (poner la carga en órbita) son exitosas; lo difícil es **recuperar el booster**.
* Entre 2010 y 2017 los resultados más frecuentes son "No attempt" y los intentos en barcaza: la empresa
  estaba aún experimentando.